## Teacher training fine-tuned

* **Objective**: Enhance model robustness and address class imbalance.

* **Description**: This notebook focuses on refining the teacher model. Starting from the best epoch (Epoch 13) of the baseline training, the approach was updated to tackle the model's inability to detect rare and spatially complex objects (like `scissors`).

* **Improvements**:
    * **Class Balancing**: Implemented an oversampling strategy specifically targeting hard and under-represented classes to force the model to learn distinctive features for objects like scissors.
    * **Advanced Augmentation**: Expanded the preprocessing pipeline to include `ColorJitter` (brightness and contrast adjustments) and `Vertical Flip` (p=0.5), increasing the model's geometric and lighting invariance.
    * Integrated a custom `_clip_bbox` funtion to enforce strict bounding box boundaries, to solve CUDA "illegal memory access" errors caused by out-of-bounds coordinates during data augmentation (especially vertical flip, where annotations could exceed image dimension), ensuring a stable training loop.

In [ ]:
import subprocess
subprocess.run(["git", "clone",
               "https://github.com/angelo4o4/sixray-kd.git",
                "/kaggle/working/sixray-kd"])

import sys
sys.path.append('/kaggle/working/sixray-kd')

!pip install -r /kaggle/working/sixray-kd/requirements.txt --quiet

Cloning into '/kaggle/working/sixray-kd'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


**Imports,device, config and paths**

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler

from src.data.dataset import SixRayDataset, collate_fn
from src.data.transforms import build_train_transforms
from src.data.labels import load_label_maps_from_file
from src.models.teacher import load_teacher
from src.engine.trainer import DetectionTrainer
from src.utils.data_utils import (
    create_train_val_split,
    class_distribution,
    print_pos_neg_balance,
    calculate_sample_weights
)
from src.utils.logger import WandbLogger, NullLogger


print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
Using device: cuda


In [ ]:
# Config
MODEL_NAME     = "PekingU/rtdetr_v2_r50vd"
USER           = "angelo"
PLATFORM       = "kaggle"
PREV_RUN_NAME  = f"01_rtdetr_teacher_baseline_{USER}"
RUN_NAME       = f"02_rtdetr_teacher_oversampling_{USER}"
CHECKPOINT_DIR = "/kaggle/working/checkpoints" if PLATFORM == "kaggle" \
                  else "/content/drive/MyDrive/DatasetAPAI/SIXray_Project/checkpoints"
RESUME_FROM    = f"/kaggle/input/datasets/angeloz404/sixray-teacher-checkpoint/{PREV_RUN_NAME}_best"  #last or best
EPOCHS         = 30
BATCH_SIZE     = 8 # before was 4
LR             = 1e-4
SEED           = 42
TRAIN_TOTAL    = 10500
VAL_POS        = 300
VAL_NEG        = 1200
USE_AUG        = True
FLIP_P         = 0.5
VFLIP_P        = 0.5
USE_WANDB      = True
WANDB_PROJECT  = "sixray-rtdetr"
EVAL_THRESHOLD = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Paths
LOCAL_EXTRACT_PATH = "/kaggle/input/datasets/angeloz404/sixray10-object-detection"
TRAIN_IMG_DIR      = os.path.join(LOCAL_EXTRACT_PATH, "train", "images")
TRAIN_JSON         = os.path.join(LOCAL_EXTRACT_PATH, "train.json")
TEST_IMG_DIR       = os.path.join(LOCAL_EXTRACT_PATH, "test", "images")
TEST_JSON          = os.path.join(LOCAL_EXTRACT_PATH, "test.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

**Model and processor**

In [ ]:
id2label, label2id, num_labels = load_label_maps_from_file(TRAIN_JSON)

id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

processor, model = load_teacher(MODEL_NAME, id2label, label2id, device=device, use_data_parallel=False)
print(f"Model loaded with {num_labels} classes: {list(id2label.values())}")

preprocessor_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

The image processor of type `RTDetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.1.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.bias to model.decoder.class_embed.2.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie class_embed.0.weight to model.decoder.class_embed.2.weight, but both are present in the checkpoints,

Model loaded with 5 classes: ['gun', 'knife', 'wrench', 'pliers', 'scissors']


**Datasets and loaders**

In [ ]:
# Adding more augmentations + handling imbalance with WeightedRandomSampler
train_transform = build_train_transforms(flip_p=FLIP_P,
                                         vflip_p=VFLIP_P,
                                         brightness=0.2,
                                         contrast=0.2,
                                         enabled=USE_AUG)

train_dataset = SixRayDataset(TRAIN_IMG_DIR, TRAIN_JSON, processor, transform=train_transform)
val_dataset   = SixRayDataset(TRAIN_IMG_DIR, TRAIN_JSON, processor, transform=None)
test_dataset  = SixRayDataset(TEST_IMG_DIR, TEST_JSON, processor, transform=None)

train_indices, val_indices, n_pos, n_neg = create_train_val_split(
    train_dataset,
    val_pos = VAL_POS,
    val_neg = VAL_NEG,
    train_total = TRAIN_TOTAL,
    seed = SEED
)

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset, val_indices)

sample_weights_tensor = calculate_sample_weights(train_dataset, train_indices, id2label)
sampler = WeightedRandomSampler(
    weights=sample_weights_tensor, 
    num_samples=len(sample_weights_tensor), 
    replacement=True
)


print(f"Full train set: {n_pos} positive / {n_neg} negative")
print(f"Train subset: {len(train_indices)} images")
print(f"Val subset:   {len(val_indices)} images")
print_pos_neg_balance("Train", train_indices, train_dataset)
print_pos_neg_balance("Val", val_indices, val_dataset)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"\nBatch image shape: {batch['pixel_values'].shape}")

Full train set: 5753 positive / 67049 negative
Train subset: 10500 images
Val subset:   1500 images
Train: 51.9% positives (5453) | 48.1% negatives (5047)
Val: 20.0% positives (300) | 80.0% negatives (1200)

Batch image shape: torch.Size([8, 3, 640, 640])


In [ ]:
# Inspecting the classes distribution
train_dist = class_distribution(train_indices, train_dataset, id2label)
val_dist   = class_distribution(val_indices, val_dataset, id2label)

print("Class distribution - Train:")
for cls, count in sorted(train_dist.items()):
    print(f"  {cls}: {count}")

print("Class distribution - Val:")
for cls, count in sorted(val_dist.items()):
    print(f"  {cls}: {count}")

print("\nWarnings:")
for cls in train_dist:
    if val_dist.get(cls, 0) < 10:
        print(f"  {cls} has only {val_dist.get(cls, 0)} val examples")

Class distribution - Train:
  gun: 3290
  knife: 1522
  pliers: 4157
  scissors: 925
  wrench: 2209
Class distribution - Val:
  gun: 175
  knife: 79
  pliers: 225
  scissors: 46
  wrench: 131

Warnings:


**Training** and **Eval**

In [ ]:
# WANDB - add checkpoint save with artifacts -
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_SILENT"] = "true"
WANDB_RUN_ID = "sixray_rtdetr_run_02"
RUN_NAME_WANDB = "rtdetr_oversampling"

logger = WandbLogger(
    project=WANDB_PROJECT,
    name=RUN_NAME_WANDB,
    id=WANDB_RUN_ID,
    resume="allow",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "train_size": len(train_indices),
        "val_size": len(val_indices),
        "use_aug": USE_AUG,
        "flip_p": FLIP_P,
        "vflip_p": VFLIP_P,
        "brightness": 0.2,
        "contrast": 0.2,
        "oversampling": True,
        "base_epoch": 13
    },
    enabled=USE_WANDB,
)

In [ ]:
trainer = DetectionTrainer(
    model=model,
    processor=processor,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    run_name=RUN_NAME,
    lr=LR,
    eval_score_threshold=EVAL_THRESHOLD,
    logger=logger,
    id2label=id2label,
    use_amp=True
)

history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, resume_from=RESUME_FROM)

Resuming from epoch 14 | best mAP so far: 0.6666
Starting training for 30 epochs (from epoch 14)
Total steps: 39390 | Warmup steps: 3939


Epoch 14/30 (Training):   0%|          | 0/1313 [00:00<?, ?it/s]

End of epoch 14 - Average Loss: 43.2111
  Val mAP: 0.4433 | mAP@50: 0.6289 | mAP@75: 0.4745
    gun: mAP = 0.7522
    knife: mAP = 0.4522
    wrench: mAP = 0.3474
    pliers: mAP = 0.3348
    scissors: mAP = 0.3301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 15/30 (Training):   0%|          | 0/1313 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 30/30 (Training):   0%|          | 0/1313 [00:00<?, ?it/s]

End of epoch 30 - Average Loss: 8.6718
  Val mAP: 0.7296 | mAP@50: 0.8462 | mAP@75: 0.7757
    gun: mAP = 0.9135
    knife: mAP = 0.8121
    wrench: mAP = 0.6567
    pliers: mAP = 0.6094
    scissors: mAP = 0.6562


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished!
Best val mAP: 0.7319 at epoch 28


In [ ]:
print(f"Val indices hash: {hash(tuple(val_indices))}")

Val indices hash: 6039922416008737244


Cells to download the weights (checkpoint)

In [ ]:
import shutil
import os

# Zip dei checkpoint
shutil.make_archive(
    "/kaggle/working/checkpoints_backup",
    "zip",
    "/kaggle/working/checkpoints"
)
print("Done:", os.path.getsize("/kaggle/working/checkpoints_backup.zip") / 1e6, "MB")

Done: 1248.622474 MB


In [ ]:
from IPython.display import Javascript
Javascript(f"window.open('/kaggle/working/checkpoints_backup.zip')")

<IPython.core.display.Javascript object>